# Data Exploration — Civic Lenses

Explore the unified contracts dataset to understand distributions, label quality, and feature relationships before modeling.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("../data/processed/unified_contracts.csv")
print(f"Dataset: {len(df)} items, {df.shape[1]} columns")
df.head(3)

## Item type and topic distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["item_type"].value_counts().plot.bar(ax=axes[0], color="#4a7c59")
axes[0].set_title("Items by Type")
axes[0].set_ylabel("Count")

df["topic"].value_counts().plot.barh(ax=axes[1], color="#4a7c59")
axes[1].set_title("Items by Topic")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

## DOGE scrutiny score distribution

This is the label used for DL model training. Top 10th percentile = "high attention", zero = "low attention".

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scores = df["doge_scrutiny_score"].dropna()
axes[0].hist(scores, bins=50, color="#4a7c59", edgecolor="white")
axes[0].axvline(scores.quantile(0.90), color="red", linestyle="--", label=f"90th pctl = {scores.quantile(0.90):.2f}")
axes[0].set_title("DOGE Scrutiny Score Distribution")
axes[0].set_xlabel("doge_scrutiny_score (savings / value)")
axes[0].set_ylabel("Count")
axes[0].legend()

# Scrutiny by topic
topic_scrutiny = df.groupby("topic")["doge_scrutiny_score"].mean().sort_values()
topic_scrutiny.plot.barh(ax=axes[1], color="#4a7c59")
axes[1].set_title("Mean DOGE Scrutiny by Topic")
axes[1].set_xlabel("Mean Scrutiny Score")

plt.tight_layout()
plt.show()

print(f"Zero scrutiny: {(scores == 0).sum()} ({(scores == 0).mean():.1%})")
print(f"Full scrutiny (1.0): {(scores == 1.0).sum()} ({(scores == 1.0).mean():.1%})")
print(f"90th percentile: {scores.quantile(0.90):.3f}")

## Contract value distribution (log scale)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

values = df["value"].dropna()
values_pos = values[values > 0]
axes[0].hist(np.log10(values_pos), bins=50, color="#4a7c59", edgecolor="white")
axes[0].set_title("Contract Value Distribution (log10)")
axes[0].set_xlabel("log10(value)")
axes[0].set_ylabel("Count")

# Value vs scrutiny
sample = df[["value", "doge_scrutiny_score"]].dropna()
sample = sample[sample["value"] > 0]
axes[1].scatter(np.log10(sample["value"]), sample["doge_scrutiny_score"], alpha=0.1, s=5, color="#4a7c59")
axes[1].set_title("Contract Value vs DOGE Scrutiny")
axes[1].set_xlabel("log10(value)")
axes[1].set_ylabel("doge_scrutiny_score")

plt.tight_layout()
plt.show()

print(f"Mean value: ${values.mean():,.0f}")
print(f"Median value: ${values.median():,.0f}")
print(f"Range: ${values.min():,.0f} — ${values.max():,.0f}")

## Feature correlations and transparency score

In [ ]:
feature_cols = ["doge_scrutiny_score", "transparency_score", "value",
                "description_length", "citizen_impact_score", "gdelt_popularity_score"]
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels([c.replace("_", "\n") for c in feature_cols], fontsize=8)
ax.set_yticklabels(feature_cols, fontsize=8)
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## Temporal distribution (deleted_date)

Used for train/val split in the DL model (cutoff: 2025-04-01).

In [ ]:
dates = pd.to_datetime(df["deleted_date"], errors="coerce").dropna()

fig, ax = plt.subplots(figsize=(12, 4))
dates.dt.to_period("W").value_counts().sort_index().plot(ax=ax, color="#4a7c59")
ax.axvline(pd.Period("2025-04-01", "W"), color="red", linestyle="--", label="Train/Val split")
ax.set_title("Contract Deletion Timeline (weekly)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

n_train = (dates < "2025-04-01").sum()
n_val = (dates >= "2025-04-01").sum()
print(f"Train (before 2025-04-01): {n_train}")
print(f"Val (after 2025-04-01): {n_val}")